# Pobieranie dzielnic i osiedli Lublina z geoportalu

Ten notebook pobiera dane o dzielnicach i osiedlach Lublina z usługi WFS (Web Feature Service) geoportalu.

In [1]:
import requests
import geopandas as gpd
from shapely.geometry import shape, box, Polygon, MultiPolygon
import json
import pandas as pd
from urllib3.exceptions import InsecureRequestWarning
import xml.etree.ElementTree as ET
import tempfile
import os

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

print("Pobieranie danych dzielnic Lublina z lokalnego WFS...\n")

wfs_url = "https://gis.lublin.eu/otwartedane/administracja/wfs"
gdf_data = None

print("="*80)
print("Pobieranie danych dzielnic i osiedli")
print("="*80 + "\n")

# Prawidłowe nazwy warstw znalezione w Capabilities
layer_names_to_try = [
    'administracja:dzielnice_granice',  # Dzielnice
    'administracja:osiedla',             # Osiedla
    'administracja:granica_miasta',      # Granica miasta
]

for layer_name in layer_names_to_try:
    print(f"Próba pobrania: {layer_name}")
    
    params = {
        'service': 'WFS',
        'version': '1.0.0',
        'request': 'GetFeature',
        'typeName': layer_name,
        'maxfeatures': '10000'
    }
    
    try:
        response = requests.get(wfs_url, params=params, timeout=30, verify=False)
        response.raise_for_status()
        
        # Parsuj XML zawartość
        print(response.content)
        root = ET.fromstring(response.content)
        
        # Sprawdź czy zawiera FeatureMembers
        feature_members = root.findall('.//{http://www.opengis.net/gml}featureMember')
        
        if len(feature_members) > 0:
            print(response.encoding)
            with open('dzielnice.gml', 'wb') as f:  # 'wb' zamiast 'w'
                f.write(response.content)
            print(f"  ✓ Otrzymano {len(feature_members)} obiektów\n")
            
            # Spróbuję sparsować GML bezpośrednio
            try:
                with tempfile.NamedTemporaryFile(mode='wb', suffix='.gml', delete=False) as f:
                    f.write(response.content)
                    print(response.text)
                    temp_file = f.name
                
                gdf_data = gpd.read_file(temp_file, encoding='utf-8')
                os.unlink(temp_file)
                
                print(f"  ✓ Powodzenie! Sparsowano {len(gdf_data)} obiektów")
                print(f"  Kolumny: {list(gdf_data.columns)}")
                print(f"  Geometria: {gdf_data.geometry.geom_type.unique()}\n")
                break
                
            except Exception as parse_err:
                if os.path.exists(temp_file):
                    os.unlink(temp_file)
                print(f"  Problem z parsowaniem GML: {str(parse_err)[:80]}\n")
        else:
            # Sprawdź czy jest błąd w XML
            service_exc = root.find('.//{http://www.opengis.net/ogc}ServiceException')
            if service_exc is not None:
                print(f"  ✗ Błąd: {service_exc.text[:80]}\n")
            else:
                print(f"  ✗ Brak obiektów w odpowiedzi\n")
                
    except Exception as e:
        print(f"  ✗ Błąd: {str(e)[:80]}\n")

# Wydruk wyniku
print(f"\n{'='*80}")
if gdf_data is not None and len(gdf_data) > 0:
    print(f"SUKCES! Pobrano dane:")
    print(f"{'='*80}")
    print(f"Liczba obiektów: {len(gdf_data)}")
    print(f"Kolumny: {list(gdf_data.columns)}")
    print(f"Typ geometrii: {gdf_data.geometry.geom_type.unique()}")
    print(f"CRS: {gdf_data.crs}")
    print(f"\nPierwsze 3 wiersze:")
    print(gdf_data.head(3))
else:
    print(f"BŁĄD: Nie udało się pobrać danych")
    print(f"{'='*80}")


Pobieranie danych dzielnic Lublina z lokalnego WFS...

Pobieranie danych dzielnic i osiedli

Próba pobrania: administracja:dzielnice_granice
b'<?xml version="1.0" encoding="UTF-8"?><wfs:FeatureCollection xmlns:wfs="http://www.opengis.net/wfs" xmlns:gml="http://www.opengis.net/gml" xmlns:administracja="administracja" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.opengis.net/wfs" xsi:schemaLocation="http://www.opengis.net/wfs http://schemas.opengis.net/wfs/1.0.0/WFS-basic.xsd administracja https://gis.lublin.eu/otwartedane/administracja/wfs?service=WFS&amp;version=1.0.0&amp;request=DescribeFeatureType&amp;typeName=administracja%3Adzielnice_granice"><gml:boundedBy><gml:Box srsName="http://www.opengis.net/gml/srs/epsg.xml#2179"><gml:coordinates decimal="." cs="," ts=" ">8391973.45,5668278.56 8407402.38,5685531.83</gml:coordinates></gml:Box></gml:boundedBy><gml:featureMember><administracja:dzielnice_granice fid="dzielnice_granice.1164"><gml:boundedBy><gml:Box srsNa

In [2]:
gdf_data.nazwa

0             Wieniawa
1             Sławinek
2      Kalinowszczyzna
3           Węglin Pd.
4            Bronowice
5         Stare Miasto
6            Czuby Pd.
7          Czechów Pd.
8            Dziesiąta
9              Wrotków
10               Głusk
11        Konstantynów
12               Felin
13         Śródmieście
14              Tatary
15                Rury
16            Kośminek
17    Hajdów - Zadębie
18        Za Cukrownią
19         Węglin Płn.
20          Abramowice
21         Czechów Pł.
22              Sławin
23            Szerokie
24           Ponikwoda
25          Czuby Płn.
26          Zemborzyce
Name: nazwa, dtype: str

In [3]:
gdf_data.crs

In [7]:
gdf_data = gdf_data.set_crs("EPSG:2179")

In [8]:
gdf_data_epsg_2180 = gdf_data.to_crs("EPSG:2180")

In [9]:
gdf_data_epsg_2180.to_file('dzielnice_lublin_epsg2180.gml', driver='GML')